##  Sales Analytics using python,sql

This project analyzes sales data using SQL queries inside Google Colab.

###  Objectives:
- Analyze monthly revenue trends
- Identify top customers
- Understand customer behavior
- Segment customers based on revenue
- Analyze city-wise performance


# Import libraries & load CSV file

In [11]:
import pandas as pd
import sqlite3

df = pd.read_csv("SALES.csv")

conn = sqlite3.connect(':memory:')
df.to_sql('sales', conn, index=False, if_exists='replace')

300

# Monthly Revenue Analysis

In [12]:
pd.read_sql("""
SELECT substr(Order_Date,1,7) AS Month,
       SUM(Revenue) AS Total_Revenue
FROM sales
GROUP BY Month
ORDER BY Month
""", conn)

,Month,Total_Revenue
0,2025-01,555900
1,2025-02,553200
2,2025-03,633100
3,2025-04,672600
4,2025-05,770300
5,2025-06,871000
6,2025-07,1134500
7,2025-08,1415000
8,2025-09,1495000
9,2025-10,1662000


# City Revenue Contribution

In [17]:
pd.read_sql("""
SELECT City,
       SUM(Revenue) AS Total_Revenue,
       ROUND(SUM(Revenue) * 100.0 /
            (SELECT SUM(Revenue) FROM sales), 2) AS Percentage
FROM sales
GROUP BY City
ORDER BY Total_Revenue DESC
""", conn)

,City,Total_Revenue,Percentage
0,Chennai,2214900,22.69
1,Bangalore,1661000,17.01
2,Delhi,1609600,16.49
3,Kolkata,1597600,16.36
4,Mumbai,1418400,14.53
5,Hyderabad,1261100,12.92


# Top Cutomer in Each City

In [18]:
pd.read_sql("""
SELECT *
FROM (
    SELECT Customer_Name,
           City,
           SUM(Revenue) AS Total_Revenue,
           ROW_NUMBER() OVER(PARTITION BY City ORDER BY SUM(Revenue) DESC) AS rn
    FROM sales
    GROUP BY Customer_Name, City
) t
WHERE rn = 1
""", conn)

,Customer_Name,City,Total_Revenue,rn
0,Deepak,Bangalore,593000,1
1,Ritu,Chennai,729000,1
2,Vikram,Delhi,566000,1
3,Asha,Hyderabad,457000,1
4,Varun,Kolkata,799000,1
5,Lakshmi,Mumbai,680000,1


# Customer Segmentation

In [16]:
pd.read_sql("""
SELECT Customer_Name,
       SUM(Revenue) AS Total_Revenue,
       CASE
           WHEN SUM(Revenue) > 500000 THEN 'High Value'
           WHEN SUM(Revenue) > 200000 THEN 'Medium Value'
           ELSE 'Low Value'
       END AS Customer_Type
FROM sales
GROUP BY Customer_Name
ORDER BY Total_Revenue DESC
""", conn)

,Customer_Name,Total_Revenue,Customer_Type
0,Varun,799000,High Value
1,Ritu,729000,High Value
2,Arun,700000,High Value
3,Lakshmi,680000,High Value
4,Priya,628800,High Value
5,Deepak,593000,High Value
6,Vikram,566000,High Value
7,Neha,565000,High Value
8,Divya,543000,High Value
9,Arvind,461000,Medium Value


## Insights

- Revenue is driven by top 10 customers (Pareto pattern)
- Some cities contribute significantly higher revenue
- Few customers generate majority of sales
- Customer segmentation helps identify high-value users
- Monthly revenue shows seasonal trends